# Training Neural Networks (MLP) with PyTorch Lightning

In [1]:
import os
import pickle
import polars as pl
import numpy as np
import pandas as pd

# Load Data

In [2]:
# input_path = './input_df' if os.path.exists('./input_df') else '/kaggle/input/js24-preprocessing-create-lags'
TRAINING = True
feature_names = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{idx}_lag_1" for idx in range(9)] + [f'symbol_id_{ind}' for ind in range(39)]
label_name = 'responder_6'
weight_name = 'weight'
train_name = "nn_input_df_with_lags.pickle"
valid_name = "nn_valid_df_with_lags.pickle"
if TRAINING and not os.path.exists(train_name):
    df = pl.scan_parquet('train.parquet').collect().to_pandas()
    valid = pl.scan_parquet("valid.parquet").collect().to_pandas()

In [3]:
df.shape, valid.shape

((21022056, 104), (1082224, 104))

In [4]:
# feature 0
# df_all.iloc[:, 5].mean(), df_all.iloc[:, 5].max() - df_all.iloc[:, 5].min()

In [5]:
# df_all.iloc[:, 8].var()

In [6]:
df.shape, valid.shape

((21022056, 104), (1082224, 104))

In [7]:
# print(list(df.columns))

In [8]:
# print(list(valid.columns))

In the valid set there is no 'symbol_id_4'!

In [9]:
# df = df_all#.iloc[-500000:, :]
# valid = valid_all#.head(500000)

Construct dummies for symbol_id

In [10]:
df = pd.get_dummies(df, columns=['symbol_id'])
valid = pd.get_dummies(valid, columns=['symbol_id'])

fill na

In [11]:
def preprocessing(df, valid, feature_names):
    
    for feature_name in feature_names:
        df[feature_name] = df[feature_name].ffill().fillna(0)
        valid[feature_name] = valid[feature_name].ffill().fillna(0)
    for feature_name in feature_names:
        df[feature_name] = df[feature_name].astype('float32')
        valid[feature_name] = valid[feature_name].astype('float32')

preprocessing(df, valid, feature_names)

In [12]:
# for feature_name in feature_names:
#     df[feature_name] = df[feature_name].ffill().fillna(0)
#     valid[feature_name] = valid[feature_name].ffill().fillna(0)

transform into float32 type

In [13]:
# for feature_name in feature_names:
#     df[feature_name] = df[feature_name].astype('float32')
#     valid[feature_name] = valid[feature_name].astype('float32')

In [14]:
df.head(5)

,id,date_id,time_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,...,symbol_id_29,symbol_id_30,symbol_id_31,symbol_id_32,symbol_id_33,symbol_id_34,symbol_id_35,symbol_id_36,symbol_id_37,symbol_id_38
0,25023058,1101,0,1.913215,0.458568,-0.052640,-0.044876,0.124223,2.819707,-0.746395,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,25023059,1101,0,4.360143,0.111398,0.500473,0.083353,0.395555,2.468412,-0.706918,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,25023060,1101,0,1.259379,0.536855,0.309993,0.015098,0.318689,2.594687,-1.157400,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,25023061,1101,0,1.498643,-0.016845,-0.104391,-0.202445,0.404921,2.384175,-0.447801,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,25023062,1101,0,1.013641,0.269194,-0.088913,-0.436358,-0.025382,2.440446,-0.579610,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
# X_train = df[ feature_names ]
# y_train = df[ label_name ]
# w_train = df[ "weight" ]
# X_valid = valid[ feature_names ]
# y_valid = valid[ label_name ]
# w_valid = valid[ "weight" ]

# X_train.shape, y_train.shape, w_train.shape, X_valid.shape, y_valid.shape, w_valid.shape

# Training Configurations

In [16]:
import os
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import (LightningDataModule, LightningModule, Trainer)
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Timer
from pytorch_lightning.loggers import WandbLogger
import wandb
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader


class custom_args():
    def __init__(self):
        self.usegpu = True
        self.gpuid = 0
        self.seed = 42
        self.model = 'nn'
        self.use_wandb = False
        self.project = 'js-xs-nn-with-lags'
        self.dname = "./input_df/"
        self.loader_workers = 4
        self.bs = 8192
        self.lr = 1e-3
        self.weight_decay = 5e-4
        self.dropouts = [0.1, 0.1] # dropout probability
        self.n_hidden = [512, 512, 256] # original 512 512 256
        self.patience = 25
        self.max_epochs = 2000
        self.N_fold = 5 # k-fold validation


my_args = custom_args()

# PyTorch Data Module Definition

In [17]:
class CustomDataset(Dataset):
    def __init__(self, df, accelerator):
        self.features = torch.FloatTensor(df[feature_names].values).to(accelerator)
        self.labels = torch.FloatTensor(df[label_name].values).to(accelerator)
        self.weights = torch.FloatTensor(df[weight_name].values).to(accelerator)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        w = self.weights[idx]
        return x, y, w


class DataModule(LightningDataModule):
    def __init__(self, train_df, batch_size, valid_df=None, accelerator='cpu'):
        super().__init__()
        self.df = train_df
        self.batch_size = batch_size
        self.dates = self.df['date_id'].unique()
        self.accelerator = accelerator
        self.train_dataset = None
        self.valid_df = None
        if valid_df is not None:
            self.valid_df = valid_df
        self.val_dataset = None

    def setup(self, fold=0, N_fold=5, stage=None):
        # Split dataset
        selected_dates = [date for ii, date in enumerate(self.dates) if ii % N_fold != fold]
        df_train = self.df.loc[self.df['date_id'].isin(selected_dates)]
        self.train_dataset = CustomDataset(df_train, self.accelerator)
        if self.valid_df is not None:
            df_valid = self.valid_df
            self.val_dataset = CustomDataset(df_valid, self.accelerator)

    def train_dataloader(self, n_workers=0):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=n_workers)

    def val_dataloader(self, n_workers=0):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=n_workers)
    


# NN Model Definition

In [18]:
# Custom R2 metric for validation
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2


class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.BatchNorm1d(in_dim))
            if i > 0:
                layers.append(nn.SiLU())
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))
            layers.append(nn.Linear(in_dim, hidden_dim))
            # layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1)) 
        layers.append(nn.Tanh())
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []

    def forward(self, x):
        return 5 * self.model(x).squeeze(-1)  

    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w  #
        loss = loss.mean()
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss

    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('val_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        """Calculate validation WRMSE at the end of the epoch."""
        y = torch.cat([x[1] for x in self.validation_step_outputs]).cpu().numpy()
        if self.trainer.sanity_checking:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
        else:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
            weights = torch.cat([x[2] for x in self.validation_step_outputs]).cpu().numpy()
            # r2_val
            val_r_square = r2_val(y, prob, weights)
            self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5,
                                                               verbose=True)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

# Create PyTorch Data Module

In [19]:
torch.cuda.is_available()

True

In [20]:
args = my_args

# checking device
device = torch.device(f'cuda:{args.gpuid}' if torch.cuda.is_available() and args.usegpu else 'cpu')
accelerator = 'gpu' if torch.cuda.is_available() and args.usegpu else 'cpu'
loader_device = 'cpu'

# Initialize Data Module

# df[feature_names] = df[feature_names].ffill().fillna(0)
# valid[feature_names] = valid[feature_names].ffill().fillna(0)
# data_module = DataModule(df, batch_size=args.bs, valid_df=valid, accelerator=loader_device)

In [21]:
device, accelerator, loader_device

(device(type='cuda', index=0), 'gpu', 'cpu')

In [22]:
data_module = DataModule(df, batch_size=args.bs, valid_df=valid, accelerator=loader_device)

# Create Model and Training

In [23]:
import gc
del df
gc.collect()
pl.seed_everything(args.seed)
for fold in range(args.N_fold):
    data_module.setup(fold, args.N_fold)
    # Obtain input dimension
    input_dim = data_module.train_dataset.features.shape[1]
    # Initialize Model
    model = NN(
        input_dim=input_dim,
        hidden_dims=args.n_hidden,
        dropouts=args.dropouts,
        lr=args.lr,
        weight_decay=args.weight_decay
    )
    # Initialize Logger
    if args.use_wandb:
        wandb_run = wandb.init(project=args.project, config=vars(args), reinit=True)
        logger = WandbLogger(experiment=wandb_run)
    else:
        logger = None
    # Initialize Callbacks
    early_stopping = EarlyStopping('val_loss', patience=args.patience, mode='min', verbose=False)
    checkpoint_callback = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1, verbose=False, filename=f"./models/nn_{fold}.model") 
    timer = Timer()
    # Initialize Trainer
    trainer = Trainer(
        max_epochs=args.max_epochs,
        accelerator=accelerator,
        devices=[args.gpuid] if args.usegpu else None,
        logger=logger,
        callbacks=[early_stopping, checkpoint_callback, timer],
        enable_progress_bar=True
    )
    # Start Training
    trainer.fit(model, data_module.train_dataloader(args.loader_workers), data_module.val_dataloader(args.loader_workers))
    # You can find trained best model in your local path
    print(f'Fold-{fold} Training completed in {timer.time_elapsed("train"):.2f}s')


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA RTX A6000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html

Epoch 27: 100%|██████████| 2054/2054 [01:11<00:00, 28.59it/s, v_num=5, val_r_square=0.00369]
Fold-0 Training completed in 1930.71s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 462 K  | train
---------------------------------------------
462 K     Trainable params
0         Non-trainable params
462 K     Total params
1.848     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Epoch 39: 100%|██████████| 2053/2053 [01:08<00:00, 30.03it/s, v_num=6, val_r_square=0.00221]
Fold-1 Training completed in 2827.45s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 462 K  | train
---------------------------------------------
462 K     Trainable params
0         Non-trainable params
462 K     Total params
1.848     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Epoch 35: 100%|██████████| 2053/2053 [01:11<00:00, 28.73it/s, v_num=7, val_r_square=0.00433]
Fold-2 Training completed in 2599.22s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 462 K  | train
---------------------------------------------
462 K     Trainable params
0         Non-trainable params
462 K     Total params
1.848     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Epoch 41: 100%|██████████| 2050/2050 [01:19<00:00, 25.74it/s, v_num=8, val_r_square=0.0053]
Fold-3 Training completed in 3118.09s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 462 K  | train
---------------------------------------------
462 K     Trainable params
0         Non-trainable params
462 K     Total params
1.848     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Epoch 29: 100%|██████████| 2057/2057 [01:15<00:00, 27.07it/s, v_num=9, val_r_square=0.00244]
Fold-4 Training completed in 2243.90s
